# Patient Flow and Transfer Analysis

Explore discharge trends, length of stay, transfer destinations, and acceptance patterns using the included synthetic records.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "generate_synthetic_data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the repository root or notebooks directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## 1. Load synthetic data


In [ ]:
DATA_DIR = ROOT / "data" / "synthetic"

df_all = pd.read_csv(DATA_DIR / "synthetic_all_disch.csv")
df_ctal = pd.read_csv(DATA_DIR / "synthetic_ctal_disch.csv")
df_ctal = df_ctal.rename(columns={"csn": "CSN"})

print("Encounter rows:", len(df_all))
print("Transfer rows:", len(df_ctal))
display(df_all.head(3))
display(df_ctal.head(3))


## 2. Data quality and cohort overview


In [ ]:
summary = pd.DataFrame({
    "rows": [len(df_all), len(df_ctal)],
    "columns": [df_all.shape[1], df_ctal.shape[1]],
    "missing_cells_pct": [
        df_all.isna().mean().mean() * 100,
        df_ctal.isna().mean().mean() * 100,
    ],
}, index=["synthetic encounters", "synthetic transfers"])

display(summary.round(2))

transfer_rate = pd.to_numeric(df_all["ip_to_other"], errors="coerce").mean()
print(f"Share of synthetic encounters marked as transfers: {transfer_rate:.1%}")


## 3. Monthly patient flow


In [ ]:
df_all["ym_dt"] = pd.to_datetime(df_all["ym"], format="%Y_%m", errors="coerce")
monthly_discharges = df_all.groupby("ym_dt").size().sort_index()
monthly_transfers = (
    df_all[df_all["ip_to_other"] == 1]
    .groupby("ym_dt")
    .size()
    .reindex(monthly_discharges.index, fill_value=0)
)
monthly_transfer_rate = monthly_transfers / monthly_discharges

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly_discharges.index, monthly_discharges.values, label="Discharges")
ax.plot(monthly_transfers.index, monthly_transfers.values, label="Transfers")
ax.set_title("Monthly Synthetic Discharges and Transfers")
ax.set_xlabel("Month")
ax.set_ylabel("Records")
ax.legend()
plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "discharges": monthly_discharges,
        "transfers": monthly_transfers,
        "transfer_rate": monthly_transfer_rate,
    }).tail()
)


## 4. Length of stay


In [ ]:
los = pd.to_numeric(df_all["InpatientLengthOfStay"], errors="coerce")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(los.dropna(), bins=35)
ax.set_title("Synthetic Inpatient Length of Stay")
ax.set_xlabel("Length of stay (days)")
plt.tight_layout()
plt.show()

los_by_transfer = (
    df_all.assign(InpatientLengthOfStay=los)
    .groupby("ip_to_other")["InpatientLengthOfStay"]
    .agg(["count", "median", "mean"])
)
los_by_transfer.index = los_by_transfer.index.map({0: "Not transferred", 1: "Transferred"})
display(los_by_transfer.round(2))


## 5. Destination hospitals


In [ ]:
destination_counts = (
    df_ctal["Patient_Transferred_to_Site"]
    .replace("", np.nan)
    .dropna()
    .value_counts()
)

fig, ax = plt.subplots(figsize=(9, 4))
destination_counts.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Synthetic Completed Transfers by Destination")
ax.set_xlabel("Completed transfers")
plt.tight_layout()
plt.show()

display(destination_counts.to_frame("completed_transfers"))


## 6. Candidate-hospital decisions


In [ ]:
first = df_ctal[["First_choice", "First_choice_status"]].rename(
    columns={"First_choice": "Hospital", "First_choice_status": "Status"}
)
second = df_ctal[["Second_choice", "Second_choice_status"]].rename(
    columns={"Second_choice": "Hospital", "Second_choice_status": "Status"}
)

decisions = pd.concat([first, second], ignore_index=True)
decisions["Hospital"] = decisions["Hospital"].fillna("").astype(str).str.strip()
decisions["Status"] = decisions["Status"].fillna("").astype(str).str.strip()
decisions = decisions[decisions["Hospital"].ne("")]

decisions["Decision"] = np.select(
    [
        decisions["Status"].str.contains("Accepted", case=False, na=False),
        decisions["Status"].str.contains("Declined", case=False, na=False),
    ],
    ["Accepted", "Declined"],
    default="Other",
)

decision_table = (
    decisions[decisions["Decision"].isin(["Accepted", "Declined"])]
    .groupby(["Hospital", "Decision"])
    .size()
    .unstack(fill_value=0)
)

for col in ["Accepted", "Declined"]:
    if col not in decision_table:
        decision_table[col] = 0

decision_table["Total"] = decision_table["Accepted"] + decision_table["Declined"]
decision_table["Acceptance Rate"] = (
    decision_table["Accepted"] / decision_table["Total"]
)

display(decision_table.sort_values("Total", ascending=False).round(3))


## 7. Transfer rate by synthetic region


In [ ]:
region_col = "HRegion" if "HRegion" in df_all.columns else "LHIN_Name"

region_summary = (
    df_all.groupby(region_col)
    .agg(
        Discharges=("PAT_ID", "size"),
        Transfers=("ip_to_other", "sum"),
    )
)
region_summary["Transfer Rate"] = (
    region_summary["Transfers"] / region_summary["Discharges"]
)

display(region_summary.sort_values("Discharges", ascending=False).round(3))


## Takeaways

The synthetic data preserves the structure needed to demonstrate:
- encounter-level patient context,
- candidate-hospital transfer decisions,
- temporal and regional features,
- hospital acceptance outcomes.

The distributions above are generated and should not be interpreted as real clinical findings.
